# Lab 06 - Integración de Datos y Streaming

**Objetivos:**
- Trabajar con diferentes formatos de datos
- Simular streaming con Auto Loader
- Integración de múltiples fuentes
- Best practices de almacenamiento

## Parte 1: Integración de Múltiples Formatos

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

print("🔧 Preparando datos de diferentes fuentes...")

In [ ]:
# Fuente 1: CSV - Catálogo de Productos
products_data = [
    {"product_id": f"P{i:03d}", "name": f"Product {i}", "category": random.choice(["Electronics", "Clothing", "Food"]), "price": round(random.uniform(10, 500), 2)}
    for i in range(1, 51)
]

df_products = spark.createDataFrame(products_data)
products_path = "/tmp/lab06/source/products.csv"

df_products.write.format("csv").option("header", "true").mode("overwrite").save(products_path)
print(f"✅ Productos guardados en CSV: {products_path}")

In [ ]:
# Fuente 2: JSON - Transacciones
transactions_data = []
for i in range(200):
    transactions_data.append({
        "transaction_id": f"TXN{i:05d}",
        "timestamp": (datetime.now() - timedelta(hours=random.randint(0, 72))).isoformat(),
        "product_id": f"P{random.randint(1, 50):03d}",
        "quantity": random.randint(1, 5),
        "customer_id": f"C{random.randint(1000, 9999)}"
    })

df_transactions = spark.createDataFrame(transactions_data)
transactions_path = "/tmp/lab06/source/transactions.json"

df_transactions.write.format("json").mode("overwrite").save(transactions_path)
print(f"✅ Transacciones guardadas en JSON: {transactions_path}")

In [ ]:
# Fuente 3: Parquet - Customer Data
customers_data = [
    {"customer_id": f"C{i}", "name": f"Customer {i}", "email": f"customer{i}@example.com", "country": random.choice(["US", "UK", "CA", "AU"])}
    for i in range(1000, 10000)
]

df_customers = spark.createDataFrame(customers_data)
customers_path = "/tmp/lab06/source/customers.parquet"

df_customers.write.format("parquet").mode("overwrite").save(customers_path)
print(f"✅ Clientes guardados en Parquet: {customers_path}")

## Parte 2: Leer Diferentes Formatos

In [ ]:
# Leer CSV
df_products_read = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(products_path)

print(f"📊 Productos CSV: {df_products_read.count()} registros")
display(df_products_read.limit(5))

In [ ]:
# Leer JSON
df_transactions_read = spark.read.format("json").load(transactions_path)

print(f"📊 Transacciones JSON: {df_transactions_read.count()} registros")
display(df_transactions_read.limit(5))

In [ ]:
# Leer Parquet
df_customers_read = spark.read.format("parquet").load(customers_path)

print(f"📊 Clientes Parquet: {df_customers_read.count()} registros")
display(df_customers_read.limit(5))

## Parte 3: Join y Enriquecimiento de Datos

In [ ]:
# Join Transacciones + Productos
df_enriched = df_transactions_read \
    .join(df_products_read, "product_id", "left") \
    .withColumn("total_amount", col("quantity") * col("price"))

print(f"✅ Datos enriquecidos: {df_enriched.count()} registros")
display(df_enriched.limit(10))

In [ ]:
# Join con Customers
df_complete = df_enriched \
    .join(df_customers_read, "customer_id", "left") \
    .select(
        "transaction_id",
        "timestamp",
        "customer_id",
        col("name").alias("customer_name"),
        "email",
        "country",
        "product_id",
        col("name").alias("product_name"),
        "category",
        "quantity",
        "price",
        "total_amount"
    )

print(f"✅ Dataset completo: {df_complete.count()} registros")
display(df_complete.limit(10))

## Parte 4: Agregaciones y Análisis

In [ ]:
# Análisis por categoría
df_by_category = df_complete \
    .groupBy("category") \
    .agg(
        count("*").alias("num_transactions"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_transaction_amount"),
        sum("quantity").alias("total_units_sold"),
        countDistinct("customer_id").alias("unique_customers")
    ) \
    .orderBy(col("total_revenue").desc())

print("📊 Análisis por Categoría:")
display(df_by_category)

In [ ]:
# Análisis por país
df_by_country = df_complete \
    .groupBy("country") \
    .agg(
        count("*").alias("num_transactions"),
        sum("total_amount").alias("total_revenue"),
        countDistinct("customer_id").alias("unique_customers")
    ) \
    .orderBy(col("total_revenue").desc())

print("📊 Análisis por País:")
display(df_by_country)

In [ ]:
# Top 10 productos
df_top_products = df_complete \
    .groupBy("product_id", "product_name", "category") \
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("total_amount").alias("total_revenue")
    ) \
    .orderBy(col("total_revenue").desc()) \
    .limit(10)

print("📊 Top 10 Productos:")
display(df_top_products)

## Parte 5: Guardar en Data Lakehouse

In [ ]:
# Guardar dataset completo particionado
lakehouse_path = "/tmp/lab06/lakehouse/transactions_complete"

df_complete \
    .withColumn("date", to_date("timestamp")) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("category", "country") \
    .save(lakehouse_path)

print(f"✅ Datos guardados en Lakehouse: {lakehouse_path}")

In [ ]:
# Guardar agregaciones (Serving Layer)
serving_path = "/tmp/lab06/serving"

# Por categoría
df_by_category.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{serving_path}/by_category")

# Por país
df_by_country.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{serving_path}/by_country")

# Top productos
df_top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{serving_path}/top_products")

print(f"✅ Agregaciones guardadas en capa Serving: {serving_path}")

## Parte 6: Simular Streaming con Auto Loader

In [ ]:
# Preparar directorio para streaming
streaming_input = "/tmp/lab06/streaming/input"
streaming_checkpoint = "/tmp/lab06/streaming/checkpoint"
streaming_output = "/tmp/lab06/streaming/output"

# Limpiar directorios anteriores
dbutils.fs.rm(streaming_input, True)
dbutils.fs.rm(streaming_checkpoint, True)
dbutils.fs.rm(streaming_output, True)

# Crear archivo inicial
batch1 = spark.range(0, 100).withColumn("value", rand() * 1000)
batch1.write.format("json").mode("overwrite").save(f"{streaming_input}/batch1")

print(f"✅ Directorio streaming preparado")

In [ ]:
# Leer con Auto Loader (cloudFiles)
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", streaming_checkpoint) \
    .load(streaming_input)

# Transformación
df_stream_processed = df_stream \
    .withColumn("processed_at", current_timestamp()) \
    .withColumn("value_category",
                when(col("value") < 250, "Low")
                .when(col("value") < 750, "Medium")
                .otherwise("High"))

# Escribir stream
query = df_stream_processed.writeStream \
    .format("delta") \
    .option("checkpointLocation", streaming_checkpoint) \
    .outputMode("append") \
    .start(streaming_output)

print("🔄 Streaming iniciado...")
print(f"   Query ID: {query.id}")
print(f"   Status: {query.status}")

In [ ]:
# Agregar más archivos al stream (simular ingesta continua)
import time

for i in range(2, 5):
    batch = spark.range(0, 50).withColumn("value", rand() * 1000)
    batch.write.format("json").mode("overwrite").save(f"{streaming_input}/batch{i}")
    print(f"✅ Batch {i} agregado")
    time.sleep(2)

print("\n⏳ Esperando procesamiento...")
time.sleep(5)

In [ ]:
# Verificar resultados
df_stream_results = spark.read.format("delta").load(streaming_output)
print(f"📊 Registros procesados: {df_stream_results.count()}")

# Ver distribución por categoría
display(df_stream_results.groupBy("value_category").count().orderBy("value_category"))

In [ ]:
# Detener streaming
query.stop()
print("⏹️  Streaming detenido")

## ✅ Lab Completado

Has aprendido:
- ✅ Trabajar con múltiples formatos (CSV, JSON, Parquet)
- ✅ Joins complejos entre datasets
- ✅ Agregaciones para análisis
- ✅ Particionado en Data Lakehouse
- ✅ Streaming con Auto Loader (cloudFiles)
- ✅ Checkpointing para procesamiento confiable

💡 **Próximos pasos**: Integrar con servicios Azure reales (Storage Account, Cosmos DB, Event Hubs) para producción.